# Receipt Parsing with DuckDB + Multimodal

Parse multiple receipt files into a structured DuckDB table using SQL.

This example demonstrates:
- **`duckdb_ext.responses_udf`** with Pydantic `response_format` → DuckDB `STRUCT`
- **Multimodal mode** — text files are read and batched; PDFs use the Files API
- **Pure SQL** for field access (`udf(col).field`) and aggregation

## 1. Setup

In [ ]:
import tempfile
from pathlib import Path

import duckdb
from pydantic import BaseModel

from openaivec.duckdb_ext import responses_udf

## 2. Define receipt schema

The Pydantic model is automatically converted to a DuckDB `STRUCT` type,
so you can access fields directly in SQL.

In [ ]:
class ReceiptItem(BaseModel):
    """A single line item on a receipt."""
    description: str
    quantity: int
    unit_price: float
    amount: float


class Receipt(BaseModel):
    """Structured representation of a receipt."""
    store_name: str
    date: str
    items: list[ReceiptItem]
    subtotal: float
    tax: float
    total: float
    payment_method: str

## 3. Create receipt files and load into DuckDB

In [ ]:
tmpdir = tempfile.mkdtemp()

receipts_raw = {
    "receipt_coffee_shop.txt": """
SUNNY CAFE
123 Main Street, Tokyo
Date: 2025-03-15

Latte                x2    ¥550    ¥1,100
Croissant            x1    ¥380    ¥380
Blueberry Muffin     x3    ¥420    ¥1,260

Subtotal: ¥2,740
Tax (10%): ¥274
Total: ¥3,014
Payment: Credit Card (Visa)
""",
    "receipt_electronics.txt": """
TECH WORLD AKIHABARA
456 Electric Town, Tokyo
Date: 2025-03-16

USB-C Cable          x2    ¥980    ¥1,960
Wireless Mouse       x1    ¥3,500  ¥3,500
Screen Protector     x3    ¥650    ¥1,950

Subtotal: ¥7,410
Tax (10%): ¥741
Total: ¥8,151
Payment: Cash
""",
    "receipt_grocery.txt": """
FRESH MART SHIBUYA
789 Center Gai, Tokyo
Date: 2025-03-17

Organic Milk 1L      x2    ¥320    ¥640
Whole Wheat Bread     x1    ¥280    ¥280
Avocado              x4    ¥198    ¥792
Salmon Fillet 200g   x2    ¥580    ¥1,160

Subtotal: ¥2,872
Tax (8%): ¥230
Total: ¥3,102
Payment: IC Card (Suica)
""",
}

file_paths = []
for name, content in receipts_raw.items():
    path = Path(tmpdir) / name
    path.write_text(content)
    file_paths.append(str(path))

conn = duckdb.connect()
conn.execute("CREATE TABLE receipts (file_path VARCHAR)")
conn.executemany("INSERT INTO receipts VALUES (?)", [(p,) for p in file_paths])
conn.sql("SELECT * FROM receipts")

## 4. Register the parsing UDF

`responses_udf` with a Pydantic `response_format` creates a UDF that returns
a DuckDB `STRUCT`. With `multimodal=True`, text files are read and batched automatically.

In [ ]:
responses_udf(
    conn,
    "parse_receipt",
    instructions=(
        "You are a receipt parser. Extract all information from the receipt "
        "into the structured format. Use numeric values without currency symbols or commas. "
        "Dates should be in YYYY-MM-DD format."
    ),
    response_format=Receipt,
    batch_size=10,
    multimodal=True,
)

## 5. Parse all receipts with SQL

Call the UDF on each file path. The result is a `STRUCT` — access fields with dot notation.

In [ ]:
conn.sql("""
    SELECT
        parse_receipt(file_path).store_name  AS store,
        parse_receipt(file_path).date        AS date,
        parse_receipt(file_path).total       AS total,
        parse_receipt(file_path).tax         AS tax,
        parse_receipt(file_path).payment_method AS payment
    FROM receipts
""")

## 6. Unnest line items

Use `UNNEST` to flatten the `items` array into individual rows.

In [ ]:
conn.sql("""
    WITH parsed AS (
        SELECT parse_receipt(file_path) AS r
        FROM receipts
    ),
    items AS (
        SELECT
            r.store_name    AS store,
            r.date          AS date,
            UNNEST(r.items) AS item,
            r.payment_method AS payment
        FROM parsed
    )
    SELECT
        store, date,
        item.description,
        item.quantity   AS qty,
        item.unit_price,
        item.amount,
        payment
    FROM items
""")

## 7. Aggregate by store

In [ ]:
conn.sql("""
    WITH parsed AS (
        SELECT parse_receipt(file_path) AS r
        FROM receipts
    )
    SELECT
        r.store_name             AS store,
        len(r.items)             AS num_items,
        r.subtotal,
        r.tax,
        r.total,
        r.payment_method         AS payment
    FROM parsed
    ORDER BY r.total DESC
""")

## 8. Cleanup

In [ ]:
conn.close()

import shutil
shutil.rmtree(tmpdir)
print("Done!")

## Notes

- **Text files** (`.txt`, `.csv`, `.md`, `.py`, etc.) are read as strings and
  processed through the **batched text path** with deduplication.
- **Binary files** (`.pdf`, `.docx`, `.xlsx`) are uploaded via the **Files API**
  and processed individually.
- The Pydantic model maps to a DuckDB `STRUCT` — access nested fields with
  dot notation and `UNNEST` arrays, all in SQL.
- For real PDF receipts, simply change the file paths — the Files API upload
  is automatic.